# 05 — Fake review detection and the ablation

Feature families plus CII into a Random Forest. Feeds **5.4**, **5.6**, and the
ablation that answers **5.10 — results against the research objectives**.


## Kernel check


In [ ]:
import sys; print(sys.executable)


## Train and evaluate

About 40 seconds.


In [ ]:
import pandas as pd
from IPython.display import Image, display
from trustlens import config, fake
res = fake.report_all()


## Model comparison

Random Forest was the pre-specified choice. Two comparisons test that rather than
assert it: gradient boosting, which also handles missing values natively, and a
logistic regression floor.


In [ ]:
res['table']


## Which families matter


In [ ]:
res['family_importance']


In [ ]:
res['top_features']


Length is 49% of total importance — the label is close to a length rule, so this is
expected rather than surprising. **CII is second at 20%**, ahead of readability and
behavioural features. `cii_ascs` is the third most important individual feature.


## The ablation

Each row adds something to the same base model and is measured on the same test set,
with bootstrapped 95% intervals.


In [ ]:
res['ablation']


In [ ]:
display(Image(filename=str(config.FIGURES/'13_ablation.png'), width=680))


### Reading it

**On the whole test set, CII adds +0.018 AUC** (0.851 → 0.869). Small, and honestly so:
the base model already has review length, which alone reaches 0.841.

**On reviews of 20+ characters, CII adds +0.209 AUC** (0.646 → 0.855). The baseline's
interval spans chance; the +CII interval does not. This is where the contribution is.

**Remove length entirely and CII adds +0.060** on the full set (0.691 → 0.751), +0.210
on longer reviews. The index is carrying real signal that length was masking.

One nuance worth reporting: SSG alone gives the largest single lift on longer reviews
(+0.199), yet the weight search in step 4 gave it only 0.05. The weights were tuned to
maximise CII's own AUC, which favours ASCS; inside the detector, SSG contributes more.
The composite is not optimal for the detection task, and saying so is more useful than
quietly re-tuning.


### Do not read F1 in the 20+ stratum

That stratum has 10 positives. F1 at a fixed 0.5 threshold moves by 0.15 when two
predictions flip, which is why several rows show F1 falling while AUC rises sharply.
AUC is threshold-free and is the metric to report here. Quote F1 from the full test set
only, and say why.


## Dropped feature families

Three of Hajek et al.'s families cannot be computed on this corpus. Each is dropped for
a stated reason rather than omitted silently.

| Family | Why |
|---|---|
| VP-based behavioural | `verified_purchase` is `True` for all 1,977 rows — constant |
| Word embedding | Skip-Gram on 1,557 reviews averaging 53 characters produces noise |
| POS tagging | No tagger for Sinhala or Singlish; median review is 32 characters, too short for a POS distribution to mean anything |

Readability is implemented inside `fake.py` rather than taken from a library — no network,
no corpus download, nothing extra for a reader to install to reproduce this.


In [ ]:
import json; json.loads((config.MODELS/'fake_meta.json').read_text())['dropped_families']


## What goes where

| Output | Section |
|---|---|
| `18_fake_model_comparison.csv` | 4.6 model selection · 5.6 technical measures |
| `19_fake_family_importance.csv`, `20_fake_top_features.csv` | 5.4 experimental results |
| `21_ablation.csv`, figure `13` | **5.10 — results against research objectives** |
